# R11-H104 - instability is an error signal: the split-and-remerge probe

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R11 identity round, resolution replay <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) + logged resolution events <br>

Registered: replaying resolution over split-back candidates reproduces <= 90% of standing merges,
and the unstable remainder is >= 3x enriched for the defect set. Reproducibility under replay as a
zero-label error detector.

## Variant that ran (stated per the batch spec)
The logged resolution events (`logs/kgf-events.jsonl`, `resolution.merge` with full posterior components
`prior x lr_description x lr_embedding x lr_cooccurrence`) ARE the replay record. Re-running the
resolver's own decision rule on each merge's logged components recomputes the posterior and re-applies
the 0.60 threshold. The decision rule is a deterministic function of the logged components (the H54
formula reconstructs the posterior at ~0 error), so every standing merge reproduces - there is no
stochastic instability in the rule itself. The registered "unstable remainder" is therefore
operationalized as PROXIMITY TO THRESHOLD: merges whose logged posterior sits within one EWMA-noise
band above 0.60 are NEAR-THRESHOLD (unstable); merges well above are COMFORTABLE. The instability
clause tests whether near-threshold merges are >= 3x enriched for adjudicated-NO (H101 labels) vs
comfortable merges.

## Outputs
- `reports/split-remerge-h104-<stamp>.json`


In [1]:
# Imports
# stdlib
import datetime, json, collections
from pathlib import Path
# third party
import numpy as np
from neo4j import GraphDatabase
from rich import print as rprint

NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # read-only neo4j2 (NOT .env live graph)
EVENTS = "../logs/kgf-events.jsonl"
BENCH = "../reports/identity-benchmark-h101-20260707-094448.json"
T_MERGE = 0.60          # posterior >= 0.60 -> merge
EWMA_ALPHA = 0.2        # control-chart smoothing for the noise band
BAR_ENRICH = 3.0        # near-threshold NO-rate / comfortable NO-rate >= 3x confirms
rprint(f"[bold]config[/bold] T_merge={T_MERGE}  ewma_alpha={EWMA_ALPHA}  bar_enrich>={BAR_ENRICH}x")


config T_merge=0.6  ewma_alpha=0.2  bar_enrich>=3.0x

## Load logged merges and confirm decision-rule reproduction\n\nEvery `resolution.merge` event carries its posterior components. Recompute the posterior from the logged components and re-apply the threshold - reproduction of the merge decision is the replay.

In [2]:
def post_from(prior, ld, le, lc):
    o = (prior / (1 - prior)) * ld * le * lc
    return o / (1 + o)

merges = []
for line in open(EVENTS):
    line = line.strip()
    if not line:
        continue
    e = json.loads(line)
    if e.get("event") == "resolution.merge":
        merges.append(e)

N = len(merges)
recon_err = [abs(post_from(m["prior"], m["lr_description"], m["lr_embedding"], m["lr_cooccurrence"]) - m["posterior"]) for m in merges]
reproduced = sum(1 for m in merges if post_from(m["prior"], m["lr_description"], m["lr_embedding"], m["lr_cooccurrence"]) >= T_MERGE)
repro_rate = reproduced / N
rprint(f"logged merges: [yellow]{N}[/yellow]  formula max error {max(recon_err):.2e}")
rprint(f"decision-rule reproduction (posterior>={T_MERGE} on replayed components): "
       f"[bold]{reproduced}/{N} = {repro_rate:.4f}[/bold]  -> the rule is deterministic; instability = threshold proximity")

post = np.array([m["posterior"] for m in merges])
rprint(f"merge posterior: min={post.min():.3f} med={np.median(post):.3f} max={post.max():.3f} "
       f"p10={np.percentile(post,10):.3f} p90={np.percentile(post,90):.3f}")


logged merges: 3492  formula max error 0.00e+00

decision-rule reproduction (posterior>=0.6 on replayed components): 3492/3492 = 1.0000  -> the rule is 
deterministic; instability = threshold proximity

merge posterior: min=0.600 med=0.829 max=0.956 p10=0.655 p90=0.915

## EWMA-noise band and the near-threshold split\n\nThe merge-posterior stream (log order) is smoothed with an EWMA; the noise band is the EWMA volatility - the root of the EWMA of squared one-step-ahead residuals. A merge is NEAR-THRESHOLD if its posterior sits in `[0.60, 0.60 + band]`, COMFORTABLE if above.

In [3]:
# EWMA of the merge-posterior stream + EWMA residual volatility (control-chart noise)
ewma = np.zeros(N); ewmv = np.zeros(N)
ewma[0] = post[0]; ewmv[0] = 0.0
for i in range(1, N):
    resid = post[i] - ewma[i - 1]                      # one-step-ahead residual
    ewma[i] = ewma[i - 1] + EWMA_ALPHA * resid
    ewmv[i] = (1 - EWMA_ALPHA) * (ewmv[i - 1] + EWMA_ALPHA * resid ** 2)
noise_band = float(np.sqrt(ewmv[-1]))                  # one EWMA-noise band (posterior units)

near_hi = T_MERGE + noise_band
is_near = (post >= T_MERGE) & (post <= near_hi)
n_near = int(is_near.sum())
n_comf = int((post > near_hi).sum())
rprint(f"[bold]EWMA-noise band[/bold] = {noise_band:.4f}  ->  near-threshold window [{T_MERGE:.3f}, {near_hi:.3f}]")
rprint(f"near-threshold merges: [yellow]{n_near}[/yellow]  comfortable merges: [yellow]{n_comf}[/yellow]")


EWMA-noise band = 0.0938  ->  near-threshold window [0.600, 0.694]

near-threshold merges: 570  comfortable merges: 2922

## Match merges to H101 adjudicated labels\n\nLogged merges carry entity ids; H101 pairs carry names (ids null). Map ids->names on neo4j2, then key both by the lowercased name pair. Adjudicated-NO = a merge whose pair the model rated distinct.

In [4]:
drv = GraphDatabase.driver(NEO4J_URI, auth=("neo4j", "kgfoundry"), notifications_min_severity="OFF")
with drv.session() as s:
    idname = {r["id"]: r["name"] for r in s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name").data()}
drv.close()

bench = json.load(open(BENCH))
label = {}   # frozenset(name_a.lower, name_b.lower) -> "YES"/"NO"/"UNCERTAIN"
for p in bench["pairs"]:
    label[frozenset((p["a"].lower(), p["b"].lower()))] = p["model_verdict"]

matched = []
for m, near in zip(merges, is_near):
    na = idname.get(m["left_id"]); nb = idname.get(m["right_id"])
    if not na or not nb:
        continue
    v = label.get(frozenset((na.lower(), nb.lower())))
    if v is None:
        continue
    matched.append({"a": na, "b": nb, "posterior": round(m["posterior"], 4),
                    "near_threshold": bool(near), "verdict": v})

n_match = len(matched)
rprint(f"logged merges matched to an H101 label: [yellow]{n_match}[/yellow] of {N}")
vc = collections.Counter(d["verdict"] for d in matched)
rprint(f"matched-merge verdicts: {dict(vc)}")


logged merges matched to an H101 label: 32 of 3492

matched-merge verdicts: {'NO': 29, 'YES': 3}

## Enrichment and verdict\n\nNO-rate among near-threshold matched merges vs comfortable matched merges. Enrichment = ratio. Bar: >= 3x confirms instability is an error signal; uniform (~1x) refutes.

In [5]:
def no_rate(group):
    g = [d for d in group if d["verdict"] in ("YES", "NO")]   # drop UNCERTAIN from the rate
    if not g:
        return None, 0, 0
    no = sum(1 for d in g if d["verdict"] == "NO")
    return no / len(g), no, len(g)

near_grp = [d for d in matched if d["near_threshold"]]
comf_grp = [d for d in matched if not d["near_threshold"]]
r_near, no_near, k_near = no_rate(near_grp)
r_comf, no_comf, k_comf = no_rate(comf_grp)

if r_near is None or r_comf is None or r_comf == 0:
    enrich = None
else:
    enrich = r_near / r_comf

thin = (k_near < 5 or k_comf < 5)
if enrich is None:
    verdict = "INCONCLUSIVE"
elif enrich >= BAR_ENRICH:
    verdict = "CONFIRMED"
else:
    verdict = "REFUTED"

rprint(f"""[bold cyan]Instability-as-error-signal[/bold cyan]
[dim]{"-"*44}[/dim]
  near-threshold matched: [yellow]{k_near}[/yellow]  NO-rate [yellow]{r_near if r_near is not None else float('nan'):.3f}[/yellow]  ({no_near} NO)
  comfortable matched:    [yellow]{k_comf}[/yellow]  NO-rate [yellow]{r_comf if r_comf is not None else float('nan'):.3f}[/yellow]  ({no_comf} NO)
  enrichment (near/comfortable): [bold yellow]{enrich if enrich is not None else float('nan'):.2f}x[/bold yellow] [dim](bar >= {BAR_ENRICH}x)[/dim]
  thin-N caveat: [{'red' if thin else 'green'}]{thin}[/]  (each arm should hold >=5 labeled merges)
  Verdict: [{'green' if verdict=='CONFIRMED' else 'red' if verdict=='REFUTED' else 'yellow'}]{verdict}[/]
""")
for d in sorted(matched, key=lambda x: x["posterior"]):
    tag = "NEAR" if d["near_threshold"] else "comf"
    col = "red" if d["verdict"] == "NO" else "green" if d["verdict"] == "YES" else "yellow"
    rprint(f"  [{col}]{d['verdict']:9s}[/] p={d['posterior']:.3f} [dim]{tag}[/dim]  {d['a'][:32]} == {d['b'][:32]}")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"split-remerge-h104-{stamp}.json"
out.write_text(json.dumps({
    "hypothesis": "R11-H104", "variant": "logged-merge decision-rule replay + threshold-proximity instability",
    "n_logged_merges": N, "decision_rule_reproduction_rate": repro_rate,
    "ewma_alpha": EWMA_ALPHA, "noise_band": noise_band,
    "near_threshold_window": [T_MERGE, near_hi],
    "n_near_threshold": n_near, "n_comfortable": n_comf,
    "n_matched_to_h101": n_match,
    "near": {"n": k_near, "no": no_near, "no_rate": r_near},
    "comfortable": {"n": k_comf, "no": no_comf, "no_rate": r_comf},
    "enrichment": enrich, "bar_enrichment": BAR_ENRICH, "thin_n": thin,
    "verdict": verdict, "matched": matched,
}, indent=2, default=str))
rprint("saved", str(out))


Instability-as-error-signal
--------------------------------------------
  near-threshold matched: 6  NO-rate 0.667  (4 NO)
  comfortable matched:    26  NO-rate 0.962  (25 NO)
  enrichment (near/comfortable): 0.69x (bar >= 3.0x)
  thin-N caveat: False  (each arm should hold >=5 labeled merges)
  Verdict: REFUTED

NO        p=0.611 NEAR  C-Flex == C-Flex+

YES       p=0.624 NEAR  Smart Ramp == SmartRamp

NO        p=0.635 NEAR  AirFit F10 for Her == AirFit F20 for Her

YES       p=0.645 NEAR  CPAP == CPAP mode

...

NO        p=0.893 comf  C-Flex+ == C-Flex

NO        p=0.906 comf  Remstar BiPAP Pro == Remstar Bipap Auto

NO        p=0.927 comf  920 oximeter == 930 Oximeter

NO        p=0.938 comf  AirSense 10 AutoSet == AirSense 11 AutoSet

saved ../reports/split-remerge-h104-20260707-101641.json